In [1]:
import os
import json
import logging
import numpy as np
import pandas as pd
import cv2 as cv
import matplotlib.pyplot as plt
from tqdm import tqdm, tqdm_notebook
from sklearn.utils import shuffle

# TensorFlow 2 imports – set protobuf implementation workaround before import
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    DepthwiseConv2D,
    Dense,
    Flatten,
    BatchNormalization,
    Dropout,
)
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping



2026-05-05 06:51:49.217924: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777963909.231815      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777963909.236221      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-05 06:51:49.254132: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
# Reproducibility
np.random.seed(1372)
tf.random.set_seed(1372)




In [3]:
class Params:
    """Loads hyper‑parameters from a JSON file."""

    def __init__(self, json_path):
        self.update(json_path)

    def save(self, json_path):
        with open(json_path, "w") as f:
            json.dump(self.__dict__, f, indent=4)

    def update(self, json_path):
        with open(json_path) as f:
            params = json.load(f)
            self.__dict__.update(params)

    @property
    def dict(self):
        return self.__dict__


def set_logger(log_path):
    logger = logging.getLogger()
    logger.setLevel(logging.INFO)
    if not logger.handlers:
        fh = logging.FileHandler(log_path)
        fh.setFormatter(logging.Formatter("%(asctime)s:%(levelname)s: %(message)s"))
        logger.addHandler(fh)
        sh = logging.StreamHandler()
        sh.setFormatter(logging.Formatter("%(message)s"))
        logger.addHandler(sh)


def save_dict_to_json(d, json_path):
    with open(json_path, "w") as f:
        json.dump({k: float(v) for k, v in d.items()}, f, indent=4)




In [4]:
# Paths
data_dir = r"../input"
train_dir = os.path.join(data_dir, r"train/train")
csv_path = os.path.join(data_dir, "train.csv")

# Load labels
df = pd.read_csv(csv_path)
df["id"] = train_dir + "/" + df["id"].astype(str)
filenames = df["id"].values
labels = df["has_cactus"].astype(np.float32).values

# Shuffle
df = shuffle(df, random_state=1372).reset_index(drop=True)
filenames = df["id"].values
labels = df["has_cactus"].astype(np.float32).values

print("sample filename:", filenames[0])
print("sample label:", labels[0])




sample filename: ../input/train/train/b61963e2b36f23c178ae7ee637b59389.jpg
sample label: 0.0


In [5]:
def _parse_function(filename, label, size):
    """Read, decode and resize an image."""
    image_string = tf.io.read_file(filename)
    image_decoded = tf.image.decode_jpeg(image_string, channels=3)
    image = tf.image.convert_image_dtype(image_decoded, tf.float32)
    resized = tf.image.resize(image, [size, size])
    return resized, label


def train_preprocess(image, label, use_random_flip):
    """Training‑time augmentations."""
    if use_random_flip:
        image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=32.0 / 255.0)
    image = tf.image.random_saturation(image, lower=0.5, upper=1.5)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label


def input_fn(is_training, filenames, labels, params):
    """Build a tf.data pipeline."""
    dataset = tf.data.Dataset.from_tensor_slices((filenames, labels))
    if is_training:
        dataset = dataset.shuffle(len(filenames))
    parse_fn = lambda f, l: _parse_function(f, l, params.image_size)
    dataset = dataset.map(parse_fn, num_parallel_calls=params.num_parallel_calls)
    if is_training:
        dataset = dataset.map(
            lambda img, lbl: train_preprocess(img, lbl, params.use_random_flip),
            num_parallel_calls=params.num_parallel_calls,
        )
        dataset = dataset.repeat()
    dataset = dataset.batch(params.batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset




In [6]:
# Create a simple params JSON (same as original)
params_json = {
    "learning_rate": 1e-3,
    "batch_size": 32,
    "num_epochs": 10,
    "image_size": 32,
    "use_random_flip": True,
    "num_labels": 2,
    "num_parallel_calls": 4,
    "save_summary_steps": 1,
}
with open("params.json", "w") as f:
    json.dump(params_json, f, indent=4)

params = Params("params.json")



In [7]:
# Train / validation split (15 % for validation)
split_idx = int(len(filenames) * 0.15)
train_fnames, val_fnames = filenames[:split_idx], filenames[split_idx:]
train_labels, val_labels = labels[:split_idx], labels[split_idx:]

train_dataset = input_fn(True, train_fnames, train_labels, params)
val_dataset = input_fn(False, val_fnames, val_labels, params)



2026-05-05 06:51:56.125361: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1777963916.125826      55 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:c6:00.0, compute capability: 8.6


In [8]:
# Model definition (unchanged architecture)
model = Sequential()
model.add(Conv2D(3, kernel_size=3, activation="relu", input_shape=(32, 32, 3)))
model.add(Conv2D(filters=16, kernel_size=3, activation="relu"))
model.add(Conv2D(filters=16, kernel_size=3, activation="relu"))
model.add(BatchNormalization())
model.add(Dropout(0.2))
model.add(DepthwiseConv2D(kernel_size=3, strides=1, padding="same", use_bias=True))
model.add(Conv2D(filters=32, kernel_size=1, activation="relu"))
model.add(Conv2D(filters=64, kernel_size=1, activation="relu"))
model.add(BatchNormalization())
model.add(Dropout(0.2))
model.add(DepthwiseConv2D(kernel_size=3, strides=2, padding="same", use_bias=True))
model.add(Conv2D(filters=128, kernel_size=1, activation="relu"))
model.add(Conv2D(filters=256, kernel_size=1, activation="relu"))
model.add(BatchNormalization())
model.add(Dropout(0.3))
model.add(DepthwiseConv2D(kernel_size=3, strides=1, padding="same", use_bias=True))
model.add(Conv2D(filters=256, kernel_size=1, activation="relu"))
model.add(BatchNormalization())
model.add(Conv2D(filters=512, kernel_size=1, activation="relu"))
model.add(BatchNormalization())
model.add(Dropout(0.3))
model.add(DepthwiseConv2D(kernel_size=3, strides=2, padding="same", use_bias=True))
model.add(Conv2D(filters=512, kernel_size=1, activation="relu"))
model.add(BatchNormalization())
model.add(Conv2D(filters=512, kernel_size=1, activation="relu"))
model.add(BatchNormalization())
model.add(Dropout(0.3))
model.add(DepthwiseConv2D(kernel_size=3, strides=1, padding="same", use_bias=True))
model.add(Conv2D(filters=1024, kernel_size=1, activation="relu"))
model.add(BatchNormalization())
model.add(Conv2D(filters=1024, kernel_size=1, activation="relu"))
model.add(BatchNormalization())
model.add(Dropout(0.3))
model.add(Flatten())
model.add(Dense(512, activation="relu"))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(256, activation="relu"))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(128, activation="elu"))
model.add(Dense(1, activation="sigmoid"))



/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
# Compile – use binary cross‑entropy and correct metric name
model.compile(
    optimizer="adam", loss=tf.keras.losses.BinaryCrossentropy(), metrics=["accuracy"]
)
model.summary()



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 30, 30, 3)      │            84 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 28, 28, 16)     │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 26, 26, 16)     │         2,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 26, 26, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 26, 26, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d                │ (None, 26, 26, 16)     │           160 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 26, 26, 32)     │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 26, 26, 64)     │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 26, 26, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 26, 26, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_1              │ (None, 13, 13, 64)     │           640 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 13, 13, 128)    │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 13, 13, 256)    │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 13, 13, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 13, 13, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_2              │ (None, 13, 13, 256)    │         2,560 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 13, 13, 256)    │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 13, 13, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 13, 13, 512)    │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 13, 13, 512)    │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 13, 13, 512)    │             

 Total params: 28,232,805 (107.70 MB)

 Trainable params: 28,222,917 (107.66 MB)

 Non-trainable params: 9,888 (38.62 KB)

In [10]:
# Callbacks – monitor the correct metric names
ckpt_path = "weights-aerial-cactus.h5"
callbacks = [
    ModelCheckpoint(
        ckpt_path, monitor="val_accuracy", verbose=1, save_best_only=True, mode="max"
    ),
    ReduceLROnPlateau(
        monitor="val_loss", factor=0.2, patience=3, verbose=1, mode="min", min_lr=1e-5
    ),
    EarlyStopping(
        monitor="val_loss",
        min_delta=1e-10,
        patience=15,
        verbose=1,
        restore_best_weights=True,
    ),
]



In [11]:
# Fit model
steps_per_epoch = max(1, (len(train_fnames) // params.batch_size))
validation_steps = max(1, (len(val_fnames) // params.batch_size))

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=params.num_epochs,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=callbacks,
    verbose=2,
)



Epoch 1/10


I0000 00:00:1777963926.983457     105 service.cc:148] XLA service 0x7efdb4002860 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777963926.983520     105 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-05-05 06:52:07.121111: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777963927.882781     105 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1777963934.755335     105 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.



Epoch 1: val_accuracy improved from -inf to 0.24992, saving model to weights-aerial-cactus.h5


66/66 - 22s - 332ms/step - accuracy: 0.8045 - loss: 0.4734 - val_accuracy: 0.2499 - val_loss: 2.1204 - learning_rate: 0.0010
Epoch 2/10

Epoch 2: val_accuracy did not improve from 0.24992
66/66 - 3s - 38ms/step - accuracy: 0.8769 - loss: 0.2986 - val_accuracy: 0.2499 - val_loss: 1.1595 - learning_rate: 0.0010
Epoch 3/10

Epoch 3: val_accuracy did not improve from 0.24992
66/66 - 2s - 36ms/step - accuracy: 0.9053 - loss: 0.2512 - val_accuracy: 0.2499 - val_loss: 0.8778 - learning_rate: 0.0010
Epoch 4/10

Epoch 4: val_accuracy did not improve from 0.24992
66/66 - 2s - 34ms/step - accuracy: 0.9171 - loss: 0.2094 - val_accuracy: 0.2499 - val_loss: 1.1445 - learning_rate: 0.0010
Epoch 5/10

Epoch 5: val_accuracy did not improve from 0.24992
66/66 - 2s - 32ms/step - accuracy: 0.9342 - loss: 0.1738 - val_accuracy: 0.2499 - val_loss: 1.0886 - learning_rate: 0.0010
Epoch 6/10

Epoch 6: val_accuracy did not improve from 0.24992

Epoch 6: ReduceLROnPlateau reducing learning rate to 0.000200000009


Epoch 9: ReduceLROnPlateau reducing learning rate to 4.0000001899898055e-05.
66/66 - 3s - 49ms/step - accuracy: 0.9555 - loss: 0.1153 - val_accuracy: 0.2649 - val_loss: 1.9670 - learning_rate: 2.0000e-04
Epoch 10/10

Epoch 10: val_accuracy improved from 0.26488 to 0.30618, saving model to weights-aerial-cactus.h5


66/66 - 4s - 57ms/step - accuracy: 0.9635 - loss: 0.0959 - val_accuracy: 0.3062 - val_loss: 2.9237 - learning_rate: 4.0000e-05
Restoring model weights from the end of the best epoch: 3.


In [12]:
# Load best weights if checkpoint was created
if os.path.exists(ckpt_path):
    model.load_weights(ckpt_path)
else:
    print("Checkpoint not found – continuing with last epoch weights.")



In [13]:
# Prepare test data and generate submission
test_df = pd.read_csv(os.path.join(data_dir, "sample_submission.csv"))
test_ids = test_df["id"].values

X_test = []
for img_id in tqdm_notebook(test_ids):
    img_path = os.path.join(data_dir, "test", img_id)
    img = cv.imread(img_path)
    if img is None:
        raise FileNotFoundError(f"Image {img_path} not found.")
    img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
    img = cv.resize(img, (32, 32))
    X_test.append(img)

X_test = np.array(X_test).astype("float32") / 255.0

# Predict probabilities
y_pred = model.predict(X_test, batch_size=32, verbose=1).reshape(-1)

# Fill submission (keep probabilities for AUC)
test_df["has_cactus"] = y_pred
submission_path = "aerial-cactus-submission.csv"
test_df.to_csv(submission_path, index=False)
print(f"Submission saved to {submission_path}")

/tmp/ipykernel_55/2783506.py:6: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for img_id in tqdm_notebook(test_ids):


  0%|          | 0/3325 [00:00<?, ?it/s]

104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step
Submission saved to aerial-cactus-submission.csv
